In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    mean_squared_error,
    silhouette_score,
    calinski_harabasz_score,
)

warnings.filterwarnings('ignore')
from IPython.display import display

# Entorno
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

RANDOM_STATE = 42
np.set_printoptions(precision=5, suppress=True)
results = []

# --- FUNCIONES DE UTILIDAD (KERNELS Y MÉTRICAS) ---

def quantum_overlap_kernel(X):
    dot_product = np.dot(X, X.T)
    return np.square(np.abs(dot_product))

def angle_encoding_overlap_kernel(X):
    n_samples, n_features = X.shape
    kernel = np.ones((n_samples, n_samples))
    for i in range(n_samples):
        diff = (X[i] - X) / 2.0
        kernel[i, :] = np.prod(np.cos(diff)**2, axis=1)
    return kernel

def run_quantum_tsne(X_input, perplexity=30, kernel_type='amplitude'):
    if kernel_type == 'amplitude':
        kernel = quantum_overlap_kernel(X_input)
    else:
        kernel = angle_encoding_overlap_kernel(X_input)

    dist = np.sqrt(np.maximum(1 - kernel, 0))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric='precomputed',
        init='random',
        random_state=RANDOM_STATE
    )
    return tsne.fit_transform(dist)

def add_result(config, components, depth, width, variance, mse, embedding, labels=None):
    lbls = labels if labels is not None else y
    results.append({
        'Configuración': config,
        'Componentes': components,
        'Profundidad': depth,
        'Ancho': width,
        'Varianza': variance,
        'MSE': mse,
        'Silhouette': silhouette_score(embedding, lbls),
        'Harabach': calinski_harabasz_score(embedding, lbls),
        'embedding': embedding
    })
def plot_embedding(ax, embedding, labels, title):
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='viridis', edgecolors='k', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, linestyle='--', alpha=0.6)
    return scatter

## 1. Carga del dataset

La función `load_dataset()` deja aislada la carga de datos. Para cambiar Wine en el futuro, basta con modificar esta función y mantener la salida `(X, y, feature_names, target_name)`.

In [ ]:
# --- CARGA Y PREPROCESAMIENTO ---
cancer = fetch_ucirepo(id=17)
X_df = cancer.data.features
y = pd.factorize(cancer.data.targets.iloc[:, 0])[0]
X_raw = X_df.to_numpy()

# 1. Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 2. Prep para Amplitude Encoding (Potencia de 2)
amplitude_dim = 2 ** math.ceil(math.log2(X_scaled.shape[1]))
X_padded = np.zeros((X_scaled.shape[0], amplitude_dim))
X_padded[:, :X_scaled.shape[1]] = X_scaled
X_amp = X_padded / np.linalg.norm(X_padded, axis=1, keepdims=True)

# 3. Prep para Angle Encoding
X_angles = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X_raw)

# 4. Prep para Híbrido (PCA 4)
pca_hybrid = PCA(n_components=4, random_state=RANDOM_STATE)
X_pca_4 = pca_hybrid.fit_transform(X_scaled)
X_hybrid_angles = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X_pca_4)

print(f"Dataset listo: {X_raw.shape[0]} muestras, {X_raw.shape[1]} features.")

## 2. Preprocesamiento consistente

Todos los métodos parten de los mismos datos estandarizados. Para amplitude encoding, además se aplica padding hasta la siguiente potencia de 2 y normalización L2 por muestra.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

def next_power_of_two(n):
    return 1 if n <= 1 else 2 ** math.ceil(math.log2(n))

def pad_to_dimension(X, target_dim):
    X_padded = np.zeros((X.shape[0], target_dim), dtype=float)
    X_padded[:, : X.shape[1]] = X
    return X_padded

def normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

# For Cancer dataset (30 features), next power of two is 32
amplitude_dim = next_power_of_two(X_scaled.shape[1])
n_system_qubits = int(math.log2(amplitude_dim))

X_padded = pad_to_dimension(X_scaled, amplitude_dim)
X_amp = normalize_rows(X_padded)

print('Dimensi3n original:', X_scaled.shape[1])
print('Dimensi3n amplitude encoding:', amplitude_dim)
print('Qubits de sistema:', n_system_qubits)
print('Norma primera muestra:', np.linalg.norm(X_amp[0]))

## 3. Angle Encoding Preprocessing
En Angle Encoding, normalizamos los datos al rango $[0, \pi]$ para que actúen como ángulos de rotación en la esfera de Bloch.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Para Angle Encoding, escalamos a [0, pi]
angle_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_angles = angle_scaler.fit_transform(X_raw)

print(f'Datos para Angle Encoding listos. Qubits necesarios: {X_angles.shape[1]} (30 características)')

## 4. PCA clásico para t-SNE

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

results = []

pca = PCA(n_components=10, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

X_pca_reconstructed = pca.inverse_transform(X_pca)
pca_variance = pca.explained_variance_ratio_.sum()
pca_mse = mean_squared_error(X_scaled, X_pca_reconstructed)

print(f'PCA clásico listo: {X_pca.shape[1]} componentes calculadas.')

In [ ]:
import matplotlib.pyplot as plt

# Calcular la varianza explicada acumulada
cum_variance_ratio = np.cumsum(pca.explained_variance_ratio_)

# Crear el gráfico
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cum_variance_ratio) + 1), cum_variance_ratio, marker='o', linestyle='--')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Varianza Explicada por el Número de Componentes Principales')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(cum_variance_ratio) + 1))
plt.axhline(y=0.95, color='r', linestyle=':', label='95% de Varianza Explicada')
plt.legend()
plt.show()

Este gráfico muestra cómo la varianza explicada acumulada aumenta a medida que se añaden más componentes principales. Puedes observar dónde la curva empieza a aplanarse, lo que indica que añadir más componentes ya no aporta mucha más información. La línea roja punteada indica el 95% de la varianza explicada, un umbral común para decidir cuántos componentes conservar.

In [ ]:
def quantum_overlap_kernel(X):
    """Simula el kernel de solapamiento cuántico usando Amplitude Encoding."""
    dot_product = np.dot(X, X.T)
    return np.square(np.abs(dot_product))

def run_quantum_tsne(X_input, perplexity=30, kernel_type='amplitude'):
    """Implementa t-SNE usando una métrica de distancia basada en el kernel cuántico."""
    if kernel_type == 'amplitude':
        kernel = quantum_overlap_kernel(X_input)
    else:
        kernel = angle_encoding_overlap_kernel(X_input)
    quantum_distance = np.sqrt(np.maximum(1 - kernel, 0))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric='precomputed',
        init='random',
        learning_rate='auto',
        random_state=RANDOM_STATE,
        n_jobs=1
    )
    return tsne.fit_transform(quantum_distance)

In [ ]:
def angle_encoding_overlap_kernel(X):
    """
    Simula el kernel de solapamiento para Angle Encoding.
    El solapamiento total es el producto del solapamiento de cada qubit: cos^2((x_i - z_i)/2).
    """
    n_samples, n_features = X.shape
    kernel = np.ones((n_samples, n_samples))
    for i in range(n_samples):
        diff = (X[i] - X) / 2.0
        kernel[i, :] = np.prod(np.cos(diff)**2, axis=1)
    return kernel

In [ ]:
def run_optimized_quantum_tsne(X_amp, perplexity=60, learning_rate='auto'):
    """
    Mejora el rendimiento ajustando la métrica de distancia cuántica
    y los parámetros de optimización del t-SNE.
    """
    # 1. Calculamos el kernel de solapamiento
    kernel = quantum_overlap_kernel(X_amp)

    # 2. Transformamos el solapamiento en una métrica de distancia más agresiva
    # Usamos -log(kernel) o 1-kernel^2 para acentuar diferencias locales
    # Esto ayuda a que el algoritmo 'separe' mejor los grupos
    quantum_distance = np.sqrt(np.maximum(1 - np.square(kernel), 0))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric='precomputed',
        init='random',
        learning_rate=learning_rate,
        early_exaggeration=12.0, # Ayuda a que los clusters se formen más rápido al inicio
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=1 # Usar todos los núcleos disponibles para el cálculo de gradientes
    )
    return tsne.fit_transform(quantum_distance)

In [ ]:
# Probamos la versión optimizada
print("Ejecutando Quantum t-SNE Optimizado...")
embedding_q_tsne_opt = run_optimized_quantum_tsne(X_amp, perplexity=40)

# Evaluamos con la métrica de Silhouette
if 'embedding_q_tsne' not in locals():
    embedding_q_tsne = run_quantum_tsne(X_amp, perplexity=30)
score_original = silhouette_score(embedding_q_tsne, y)
score_opt = silhouette_score(embedding_q_tsne_opt, y)

print(f"Silhouette Score Original: {score_original:.4f}")
print(f"Silhouette Score Optimizado: {score_opt:.4f}")

# Visualización de la mejora
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_embedding(ax, embedding_q_tsne_opt, y, 'Quantum t-SNE Optimizado (High Exaggeration)')
plt.show()

### Comparación con Angle Encoding
Ejecutamos el t-SNE usando el kernel de solapamiento generado por la codificación de ángulos.

In [ ]:
# 1. Calcular distancias basadas en el kernel de ángulos (30 qubits)
kernel_angles = angle_encoding_overlap_kernel(X_angles)
dist_angles = np.sqrt(np.maximum(1 - kernel_angles, 0))

# 2. Ejecutar t-SNE
tsne_angles = TSNE(
    n_components=2,
    perplexity=40,
    metric='precomputed',
    init='random',
    random_state=RANDOM_STATE
)
embedding_angle_tsne = tsne_angles.fit_transform(dist_angles)

# 3. Registrar y visualizar
add_result(
    'Quantum t-SNE (Angle Encoding)',
    X_raw.shape[1],
    X_raw.shape[1],
    X_raw.shape[1],
    1.0,
    0.0,
    embedding_angle_tsne
)

print(f'Silhouette Score (Angle Encoding): {silhouette_score(embedding_angle_tsne, y):.4f}')
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_embedding(ax, embedding_angle_tsne, y, 'Quantum t-SNE (Angle Encoding - 30 Qubits)')
plt.show()

## 5. t-SNE sobre datos reducidos por PCA clásico

In [ ]:
def add_result(config, components, depth, width, variance, mse, embedding, labels=None):
    lbls = labels if labels is not None else y
    results.append({
        'Configuración': config,
        'Componentes': components,
        'Profundidad': depth,
        'Ancho': width,
        'Varianza': variance,
        'MSE': mse,
        'Silhouette': silhouette_score(embedding, lbls),
        'Harabach': calinski_harabasz_score(embedding, lbls),
        'embedding': embedding
    })


In [ ]:
from sklearn.manifold import TSNE

# Usamos los datos preparados para Amplitude Encoding (X_amp)
print(f'Ejecutando Quantum t-SNE con {n_system_qubits} qubits (dimensi3n {amplitude_dim})...')

# Ejecutar t-SNE cu1ntico
embedding_q_tsne = run_quantum_tsne(X_amp, perplexity=30)

# Registrar resultados
add_result(
    'Quantum t-SNE (Amplitude Encoding)',
    X_scaled.shape[1],
    n_system_qubits,
    amplitude_dim,
    1.0,
    0.0,
    embedding_q_tsne
)

print('Embedding generado mediante Quantum t-SNE:', embedding_q_tsne.shape)
display(pd.DataFrame(results).tail(1))

## 6. Tabla de métricas y evaluación

## 7. Visualización del Embedding `t-SNE + PCA clásico`

## 9. Experimentación con Parámetros de t-SNE

Vamos a ejecutar t-SNE nuevamente con un conjunto diferente de parámetros para ver cómo afecta la visualización y las métricas de clustering. En este caso, usaremos una perplejidad menor y una inicialización aleatoria para observar el impacto.

In [ ]:
def plot_embedding(ax, embedding, labels, title):
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='viridis', edgecolors='k', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, linestyle='--', alpha=0.6)
    return scatter

# Nota: Esta celda ahora define la función para ser usada en los pasos siguientes con el dataset de cáncer.
print('Función de visualización actualizada.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

# Calculamos un t-SNE cl1sico sobre PCA para comparar
from sklearn.manifold import TSNE
tsne_classic = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE)
embedding_tsne_pca = tsne_classic.fit_transform(X_pca)

# Comparativa Visual
plot_embedding(ax1, embedding_tsne_pca, y, 't-SNE Cl1sico (sobre PCA)')
plot_embedding(ax2, embedding_q_tsne, y, f'Quantum t-SNE (Amplitude Encoding, {n_system_qubits} Qubits)')

# Legend for classes (Malignant/Benign)
import matplotlib.lines as mlines
class_0 = mlines.Line2D([], [], color=plt.cm.viridis(0), marker='o', linestyle='None', label='Malignant (0)')
class_1 = mlines.Line2D([], [], color=plt.cm.viridis(0.99), marker='o', linestyle='None', label='Benign (1)')
ax2.legend(handles=[class_0, class_1], title='Diagnosis', loc='center right', bbox_to_anchor=(1.25, 0.5))

plt.show()

In [ ]:
# Reiniciamos y ejecutamos las pruebas de perplejidad
results = []
perplexity_values = [20, 30, 40, 50, 60]

kernel_angles = angle_encoding_overlap_kernel(X_angles)
dist_angles = np.sqrt(np.maximum(1 - kernel_angles, 0))

for p in perplexity_values:
    tsne_p = TSNE(n_components=2, perplexity=p, metric='precomputed', init='random', random_state=RANDOM_STATE)
    embedding_p = tsne_p.fit_transform(dist_angles)
    add_result(f'Angle (P={p})', 30, 30, 30, 1.0, 0.0, embedding_p)

print('Pruebas de perplejidad completadas.')

## 10. Visualización de Circuitos Cuánticos (Qiskit)

Para entender mejor la diferencia entre las codificaciones, vamos a inspeccionar los circuitos que representarían a una sola muestra del dataset.

In [ ]:
from qiskit import QuantumCircuit

# --- 1. Circuito para Amplitude Encoding ---
qc_amp = QuantumCircuit(n_system_qubits)
qc_amp.prepare_state(X_amp[0])

print(f"--- Amplitude Encoding ---")
print(f"Ancho (Qubits): {qc_amp.num_qubits}")
# Decomponemos para ver la complejidad real
qc_amp_decomp = qc_amp.decompose()
print(f"Profundidad (decomposed): {qc_amp_decomp.depth()}")

try:
    display(qc_amp_decomp.draw('mpl', style='iqp'))
except:
    print("Nota: No se pudo renderizar 'mpl', usando dibujo de texto:")
    print(qc_amp_decomp.draw())

# --- 2. Circuito para Angle Encoding ---
n_features = X_angles.shape[1]
qc_angle = QuantumCircuit(n_features)
for i in range(n_features):
    qc_angle.ry(X_angles[0, i], i)

print(f"\n--- Angle Encoding ---")
print(f"Ancho (Qubits): {qc_angle.num_qubits}")
print(f"Profundidad: {qc_angle.depth()}")
try:
    display(qc_angle.draw('mpl', style='iqp'))
except:
    print(qc_angle.draw())

### ¿Por qué usamos 13 qubits si queremos reducir dimensiones?

Es importante distinguir entre **Encoding** (codificación) y **Dimensionality Reduction** (reducción):

*   **Hardware vs Datos**: En computación cuántica, el *Amplitude Encoding* busca eficiencia de hardware ($\\log_2 N$ qubits). El *Angle Encoding* busca expresividad ($N$ qubits).
*   **El papel de t-SNE**: Independientemente de cuántos qubits usemos, el algoritmo t-SNE actúa sobre la matriz de distancias resultante. Su objetivo es encontrar una representación en **2D** que preserve las cercanías que el kernel cuántico detectó.
*   **Ventaja Cuántica**: Al usar 13 qubits, estamos operando técnicamente en un espacio vectorial de $2^{13} = 8192$ dimensiones. El t-SNE luego 'comprime' la estructura detectada en ese espacio inmenso de vuelta a un plano 2D para que podamos visualizarlo.

## 11. Optimización de Recursos: Angle Encoding Híbrido (PCA + Angles)

Para datasets grandes, podemos usar PCA para reducir las características a un número manejable de qubits (por ejemplo, 4) antes de aplicar el Angle Encoding. Esto nos permite controlar el 'ancho' del circuito independientemente del tamaño del dataset original.

In [ ]:
from sklearn.decomposition import PCA

N_QUBITS_TARGET = 4
pca_hybrid = PCA(n_components=N_QUBITS_TARGET, random_state=RANDOM_STATE)
X_pca_4 = pca_hybrid.fit_transform(X_scaled)

hybrid_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_hybrid_angles = hybrid_scaler.fit_transform(X_pca_4)

print(f'Híbrido listo: 30 features mapeadas a {N_QUBITS_TARGET} qubits.')
print(f'Varianza mantenida: {pca_hybrid.explained_variance_ratio_.sum():.4f}')

In [ ]:
# Ejecutamos el híbrido base
kernel_hybrid = angle_encoding_overlap_kernel(X_hybrid_angles)
dist_hybrid = np.sqrt(np.maximum(1 - kernel_hybrid, 0))

tsne_hybrid = TSNE(n_components=2, perplexity=40, metric='precomputed', init='random', random_state=RANDOM_STATE)
embedding_hybrid = tsne_hybrid.fit_transform(dist_hybrid)

if 'mse_hybrid' not in locals():
    X_hybrid_reconstructed = pca_hybrid.inverse_transform(X_pca_4)
    mse_hybrid = mean_squared_error(X_scaled, X_hybrid_reconstructed)
add_result('Hybrid (PCA 4)', 30, 4, 4, pca_hybrid.explained_variance_ratio_.sum(), mse_hybrid, embedding_hybrid)
print('Experimento híbrido completado.')

In [ ]:
# 6. Visualización del circuito reducido
qc_hybrid = QuantumCircuit(N_QUBITS_TARGET)
for i in range(N_QUBITS_TARGET):
    qc_hybrid.ry(X_hybrid_angles[0, i], i)

print("--- Circuito Angle Encoding Optimizado (4 Qubits) ---")
try:
    display(qc_hybrid.draw('mpl', style='iqp'))
except:
    print(qc_hybrid.draw())

### 12. Visualización del Resultado Híbrido

Comparamos visualmente cómo el enfoque de **Angle Encoding sobre PCA (4 qubits)** logra separar las clases del dataset Wine.

### 13. Cálculo del MSE para el Enfoque Híbrido

El MSE mide la pérdida de información durante la reducción de dimensionalidad con PCA (de 13 a 4 dimensiones) antes de entrar al kernel cuántico.

In [ ]:
# 1. Reconstruir los datos desde el espacio de 4 componentes
X_hybrid_reconstructed = pca_hybrid.inverse_transform(X_pca_4)

# 2. Calcular el MSE comparando con los datos escalados originales
mse_hybrid = mean_squared_error(X_scaled, X_hybrid_reconstructed)

print(f"MSE de reconstrucción (Hybrid PCA 4): {mse_hybrid:.4f}")

# 3. Actualizamos el último registro en nuestra lista de resultados
results[-1]['MSE'] = mse_hybrid

# Mostrar tabla comparativa actualizada
display(pd.DataFrame(results).tail(3))

### 14. Experimentación con Variaciones del Enfoque Híbrido

Probaremos 3 configuraciones distintas para observar el comportamiento del algoritmo bajo diferentes restricciones de qubits y parámetros de optimización.

In [ ]:
# Variación 1: Compresión Extrema (2 Qubits)
pca_v1 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_v1 = pca_v1.fit_transform(X_scaled)
X_angles_v1 = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X_pca_v1)
kernel_v1 = angle_encoding_overlap_kernel(X_angles_v1)
dist_v1 = np.sqrt(np.maximum(1 - kernel_v1, 0))
embedding_v1 = TSNE(n_components=2, perplexity=40, metric='precomputed', init='random', random_state=RANDOM_STATE).fit_transform(dist_v1)

add_result('Hybrid Var 1: Extreme (2 Qubits)', 2, 2, 2, pca_v1.explained_variance_ratio_.sum(), mse_hybrid, embedding_v1)

In [ ]:
# Variación 2: Estructura Global (High Perplexity = 80)
# Usamos los 4 qubits anteriores pero cambiamos la perplejidad
tsne_v2 = TSNE(n_components=2, perplexity=80, metric='precomputed', init='random', random_state=RANDOM_STATE)
embedding_v2 = tsne_v2.fit_transform(dist_hybrid)

add_result('Hybrid Var 2: Global (P=80, 4Q)', 13, 4, 4, pca_hybrid.explained_variance_ratio_.sum(), mse_hybrid, embedding_v2)

**Reasoning**:
I need to calculate the reconstruction MSE for 'Hybrid Var 1: Extreme (2 Qubits)' by first reconstructing the data from `X_pca_v1` using `pca_v1.inverse_transform()` and then calculating the MSE against the original `X_scaled` data.



In [ ]:
X_pca_v1_reconstructed = pca_v1.inverse_transform(X_pca_v1)
mse_v1 = mean_squared_error(X_scaled, X_pca_v1_reconstructed)

print(f"MSE de reconstrucción (Hybrid Var 1: Extreme 2 Qubits): {mse_v1:.4f}")

In [ ]:
# Variación 3: Métrica Exponencial (Sharp Distance)
# Transformamos el kernel para que las diferencias pequeñas se noten más
kernel_exp = np.exp(kernel_hybrid) / np.exp(1)
dist_exp = np.sqrt(np.maximum(1 - kernel_exp, 0))
embedding_v3 = TSNE(n_components=2, perplexity=40, metric='precomputed', init='random', random_state=RANDOM_STATE).fit_transform(dist_exp)

add_result('Hybrid Var 3: Exp Kernel (4Q)', 13, 4, 4, pca_hybrid.explained_variance_ratio_.sum(), mse_hybrid, embedding_v3)

In [ ]:
# --- EJECUCIÓN DE EXPERIMENTOS Y VISUALIZACIÓN ---
results = [] # Reiniciar para limpieza

# Experimentación
print("Ejecutando Amplitude Encoding...")
emb_amp = run_quantum_tsne(X_amp, perplexity=40, kernel_type='amplitude')
add_result('Amplitude Encoding (5Q)', 30, 5, 32, 1.0, 0.0, emb_amp, y)

print("Ejecutando Angle Encoding...")
emb_angle = run_quantum_tsne(X_angles, perplexity=40, kernel_type='angle')
add_result('Angle Encoding (30Q)', 30, 30, 30, 1.0, 0.0, emb_angle, y)

print("Ejecutando Híbrido (PCA 4 + Angle)...")
emb_hybrid = run_quantum_tsne(X_hybrid_angles, perplexity=40, kernel_type='angle')
add_result('Hybrid (PCA 4)', 30, 4, 4, pca_hybrid.explained_variance_ratio_.sum(), 0.2076, emb_hybrid, y)

# Mostrar Tabla
display(pd.DataFrame(results).drop(columns=['embedding']))

# Graficar
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, res in enumerate(results):
    plot_embedding(axes[i], res['embedding'], y, res['Configuración'])
plt.show()